# Optimization

## 🎯 Introduction

**Definition**: Optimization is choosing the **best option** from a set of possible options.

**Goal**: Find the solution that maximizes or minimizes some objective, given constraints.

**Examples**:
- Finding the shortest delivery route
- Scheduling exams without conflicts
- Minimizing production cost

---

# Local Search

## 🗺️ What is Local Search?

**Definition**: A search algorithm that maintains a **single node** and searches by moving to a neighboring node.

**Key difference from previous search**:
- Previous search (maze): Find the *quickest path* to a goal
- Local search: Find the *best answer* to a question (not interested in the path)

**Trade-off**: Often finds a "good enough" answer rather than the globally optimal one — but saves massive computational power.

---

## 🏥 Example: Hospital Placement

**Problem**: Place 2 hospitals to minimize total distance from 4 houses.

**Cost** = sum of Manhattan distances from each house to its nearest hospital.

**State** = any configuration of house/hospital positions.

**Goal** = find the configuration with the lowest cost.

---

## 📐 Key Terms

| Term | Definition |
|---|---|
| **Objective Function** | Function to *maximize* the value of a solution |
| **Cost Function** | Function to *minimize* the cost of a solution |
| **Current State** | The state currently being considered |
| **Neighbor State** | A state reachable from the current state in one step |

**Neighbor state (hospital example)**: Moving one hospital one step in any direction.

![State-Space Landscape](./images/statespace.png)

---

## 🧗 Hill Climbing

**Idea**: From the current state, look at all neighbor states and move to the best one. Stop when no neighbor is better.

### Pseudocode

```
function Hill-Climb(problem):
    current = initial state of problem
    repeat:
        neighbor = best valued neighbor of current
        if neighbor not better than current:
            return current
        current = neighbor
```

**Notes**:
- Uses a cost function → prefer *lower* value
- Uses an objective function → prefer *higher* value
- Algorithm is **greedy**: only moves if a neighbor is strictly better

---

## ⚠️ Local vs. Global Minima/Maxima

**The core problem**: Hill climbing can get stuck.

| Term | Meaning |
|---|---|
| **Global maximum** | Highest value state across *all* states |
| **Local maximum** | Higher than neighbors, but not globally best |
| **Global minimum** | Lowest value state across *all* states |
| **Local minimum** | Lower than neighbors, but not globally best |
| **Flat local max/min** | Plateau of equal values; neighbors are worse |
| **Shoulder** | Plateau where some neighbors are better, some worse |

![Maxima](./images/maxima.png)

![Minima](./images/minima.png)

![Flat Local Maximum/Minimum and Shoulder](./images/flatshoulder.png)

**Problem**: From a local optimum, no neighbor looks better → algorithm halts prematurely.

---

## 🔀 Hill Climbing Variants

All variants still risk getting stuck in local optima — but they reduce the chance.

| Variant | Strategy |
|---|---|
| **Steepest-ascent** | Choose the *highest-valued* neighbor (standard) |
| **Stochastic** | Choose *randomly* from higher-valued neighbors |
| **First-choice** | Choose the *first* higher-valued neighbor found |
| **Random-restart** | Run hill climbing many times from random starts; return best result |
| **Local beam search** | Keep track of *k* best states simultaneously |

**Random-restart** is especially useful — multiple independent runs increase the chance of hitting the global optimum.

---

## 🌡️ Simulated Annealing

**Motivation**: Break out of local optima by occasionally accepting *worse* states.

**Metaphor**: Heating metal then cooling slowly (annealing) makes it stronger. High temperature → more random; low temperature → more stable.

### Pseudocode

```
function Simulated-Annealing(problem, max):
    current = initial state of problem
    for t = 1 to max:
        T = Temperature(t)
        neighbor = random neighbor of current
        ΔE = how much better neighbor is than current
        if ΔE > 0:
            current = neighbor
        with probability e^(ΔE/T) set current = neighbor
    return current
```

### Key Mechanics

- **T (temperature)**: High early on → low later. Controls willingness to accept worse states.
- **ΔE**: How much better the neighbor is. If negative, neighbor is worse.
- **Acceptance probability**: $e^{\Delta E / T}$
  - More negative ΔE → lower probability (worse neighbors less likely accepted)
  - Higher T → probability closer to 1 (early on, more likely to accept bad moves)

### Traveling Salesman Problem (TSP)

A classic example where simulated annealing shines:

- **Task**: Visit all cities exactly once, using the shortest total distance.
- **Scale**: 10 cities = 10! = 3,628,800 possible routes → brute force is infeasible.
- **Neighbor state**: Swap two edges in the current route.
- Simulated annealing finds a *good enough* solution at a fraction of the cost.

---

## 💻 Source: `hospitals/hospitals.py`

Implements the hospital placement problem using **hill climbing** and **random restart**.

**Key classes/methods**:

| Method | What it does |
|---|---|
| `Space.__init__` | Sets up grid with houses and hospitals |
| `hill_climb()` | Steepest-ascent hill climbing; moves to best neighbor until no improvement |
| `random_restart(max)` | Calls `hill_climb()` `max` times from random starts; keeps the best result |
| `get_cost(hospitals)` | Sums Manhattan distances from each house to its nearest hospital |
| `get_neighbors(row, col)` | Returns 4-directional neighbors (up/down/left/right) not occupied |

**How it works**:
1. Hospitals are placed randomly.
2. At each step, every possible move of every hospital is evaluated.
3. The move with the lowest cost is taken; ties broken randomly.
4. Stops when no neighbor improves cost.

```python
s = Space(height=10, width=20, num_hospitals=3)
hospitals = s.hill_climb(log=True)       # single run
hospitals = s.random_restart(20, log=True)  # 20 restarts
```

---

# Linear Programming

## 📐 What is Linear Programming?

**Definition**: Optimize a **linear equation** subject to linear constraints.

### Components

| Component | Form |
|---|---|
| **Cost function** (minimize) | $c_1 x_1 + c_2 x_2 + \ldots + c_n x_n$ |
| **Inequality constraint** | $a_1 x_1 + a_2 x_2 + \ldots + a_n x_n \leq b$ |
| **Equality constraint** | $a_1 x_1 + a_2 x_2 + \ldots + a_n x_n = b$ |
| **Variable bounds** | $l_i \leq x_i \leq u_i$ |

---

## 🏭 Example: Machine Scheduling

**Setup**:
- Machine $X_1$: costs \$50/hr, needs 5 labor units/hr, produces 10 output units/hr
- Machine $X_2$: costs \$80/hr, needs 2 labor units/hr, produces 12 output units/hr
- Total labor budget: 20 units
- Minimum output needed: 90 units

**Formulation**:

| | Formula |
|---|---|
| Minimize | $50x_1 + 80x_2$ |
| Labor constraint | $5x_1 + 2x_2 \leq 20$ |
| Output constraint | $-10x_1 - 12x_2 \leq -90$ (multiply by -1 to flip ≥ to ≤) |

**Algorithms**: Simplex, Interior-Point (handled by libraries like `scipy`).

---

## 💻 Source: `production/production.py`

Solves the machine scheduling problem above using `scipy.optimize.linprog`.

```python
import scipy.optimize

result = scipy.optimize.linprog(
    [50, 80],                      # cost function coefficients
    A_ub=[[5, 2], [-10, -12]],     # inequality constraint matrix
    b_ub=[20, -90],                # inequality constraint bounds
)

if result.success:
    print(f"X1: {round(result.x[0], 2)} hours")
    print(f"X2: {round(result.x[1], 2)} hours")
```

- `linprog` minimizes by default — pass the cost coefficients directly.
- Flip `≥` constraints to `≤` by multiplying both sides by `-1` before passing.

---

# Constraint Satisfaction

## 🧩 What is a CSP?

**Constraint Satisfaction Problem (CSP)**: Assign values to variables such that all constraints are satisfied.

### Properties

| Property | Description |
|---|---|
| **Variables** | $\{x_1, x_2, \ldots, x_n\}$ |
| **Domains** | $\{D_1, D_2, \ldots, D_n\}$ — possible values for each variable |
| **Constraints** | Rules that limit which combinations of values are valid |

### Types of Constraints

| Type | Definition | Example |
|---|---|---|
| **Hard** | Must be satisfied in any valid solution | No two exams same day for same student |
| **Soft** | Preference; used to rank solutions | Prefer morning exams |
| **Unary** | Involves one variable | $A \neq \text{Monday}$ |
| **Binary** | Involves two variables | $A \neq B$ |

---

## 📅 Example: Exam Scheduling

**Problem**: 4 students each take 3 courses from {A,B,C,D,E,F,G}. Schedule exams on Mon/Tue/Wed with no student having two exams the same day.

- **Variables**: Courses A–G
- **Domain**: {Monday, Tuesday, Wednesday}
- **Constraints**: Pairs of courses sharing a student must have different days

The constraints form a **graph** where nodes = courses, edges = "can't be on same day."

![Constraint Graph](./images/constraintsatisfaction2.png)

---

## 🔵 Node Consistency

**Definition**: A variable is node-consistent if all values in its domain satisfy its **unary** constraints.

**Example**:
- Domain of A = {Mon, Tue, Wed}, constraint: $A \neq \text{Mon}$
- Not node-consistent; remove Mon → Domain = {Tue, Wed} → node-consistent ✅

---

## 🔗 Arc Consistency

**Definition**: Variable X is arc-consistent with respect to Y if for every value in X's domain, there exists *at least one* compatible value in Y's domain.

**Goal**: Remove values from X's domain until arc-consistency holds.

### Revise Algorithm

```
function Revise(csp, X, Y):
    revised = false
    for x in X.domain:
        if no y in Y.domain satisfies constraint for (X, Y):
            delete x from X.domain
            revised = true
    return revised
```

### AC-3 Algorithm (Full Arc Consistency)

```
function AC-3(csp):
    queue = all arcs in csp
    while queue non-empty:
        (X, Y) = Dequeue(queue)
        if Revise(csp, X, Y):
            if X.domain is empty:
                return false          # problem is unsolvable
            for each Z in X.neighbors - {Y}:
                Enqueue(queue, (Z, X))
    return true
```

**How it works**:
1. Start with all arcs in a queue.
2. For each arc (X, Y): run `Revise` to enforce consistency.
3. If X's domain shrank, all of X's other neighbors must be re-checked.
4. If any domain becomes empty → no solution exists.

**Limitation**: AC-3 alone does not always solve the CSP — it simplifies it, but cannot handle all interactions among multiple variables.

---

## 🔙 Backtracking Search

**Idea**: Treat CSP as a search problem. Assign variables one at a time; backtrack when a constraint is violated.

### Pseudocode

```
function Backtrack(assignment, csp):
    if assignment complete:
        return assignment
    var = Select-Unassigned-Var(assignment, csp)
    for value in Domain-Values(var, assignment, csp):
        if value consistent with assignment:
            add {var = value} to assignment
            result = Backtrack(assignment, csp)
            if result ≠ failure:
                return result
            remove {var = value} from assignment
    return failure
```

![Backtracking Example](./images/backtracking.png)

**How it works**:
1. If all variables are assigned and consistent → return solution.
2. Pick an unassigned variable.
3. Try each value in its domain; if consistent, recurse.
4. If recursion fails, undo the assignment and try the next value.
5. If all values fail → return failure (triggers backtracking in the caller).

---

## 🧠 Inference: Maintaining Arc Consistency (MAC)

**Idea**: After each assignment, run AC-3 to prune domains *before* continuing. Catches failures earlier.

```
function Backtrack(assignment, csp):
    ...
    add {var = value} to assignment
    inferences = Inference(assignment, csp)   ← runs AC-3
    if inferences ≠ failure:
        add inferences to assignment
        result = Backtrack(assignment, csp)
        if result ≠ failure:
            return result
    remove {var = value} and inferences from assignment
    ...
```

**Benefit**: Domains shrink after each assignment → fewer dead ends, less backtracking.

---

## 💡 Heuristics for Backtracking

Choosing *which variable* and *which value* to try first can dramatically speed up search.

### Variable Ordering

| Heuristic | Rule | Intuition |
|---|---|---|
| **MRV** (Minimum Remaining Values) | Pick variable with *fewest* values in its domain | If it's going to fail, find out early |
| **Degree** | Pick variable with *most* constraints on other variables | One assignment constrains many others |

Use **MRV first**; break ties with **Degree**.

![Minimum Remaining Values](./images/mrv.png)

![Degree Heuristic](./images/degree.png)

### Value Ordering

| Heuristic | Rule | Intuition |
|---|---|---|
| **Least Constraining Value** | Pick value that rules out the *fewest* choices for neighbors | Keep options open for future variables |

![Least Constraining Value](./images/leastconstrainingvalue.png)

**Key insight**: For variables, choose the *most* constrained (MRV/Degree). For values, choose the *least* constraining. This focuses the search where it matters while preserving flexibility everywhere else.

---

## 💻 Source: `scheduling/schedule0.py` — Naive Backtracking

Solves the exam scheduling CSP from scratch using plain backtracking (no heuristics, no inference).

```python
VARIABLES = ["A", "B", "C", "D", "E", "F", "G"]
CONSTRAINTS = [("A","B"), ("A","C"), ...]  # pairs that can't share a day

def backtrack(assignment):
    if len(assignment) == len(VARIABLES):
        return assignment                  # complete
    var = select_unassigned_variable(assignment)
    for value in ["Monday", "Tuesday", "Wednesday"]:
        new_assignment = assignment.copy()
        new_assignment[var] = value
        if consistent(new_assignment):
            result = backtrack(new_assignment)
            if result is not None:
                return result
    return None                            # failure → backtrack
```

- `select_unassigned_variable`: picks variables in order (no MRV/Degree).
- `consistent`: checks that no two constrained variables share the same day.

---

## 💻 Source: `scheduling/schedule1.py` — Library-Based CSP

Solves the same problem using the **`python-constraint`** library — no manual backtracking needed.

```python
from constraint import *

problem = Problem()
problem.addVariables(["A","B","C","D","E","F","G"], ["Monday","Tuesday","Wednesday"])

for x, y in CONSTRAINTS:
    problem.addConstraint(lambda x, y: x != y, (x, y))

for solution in problem.getSolutions():
    print(solution)
```

- The library handles backtracking, inference, and heuristics internally.
- `getSolutions()` returns *all* valid assignments.
- Compare with `schedule0.py`: same logic, but the library makes it ~10x shorter.

---

## 📝 Key Takeaways

### Local Search
✅ Maintains a single state; moves to better neighbors  
✅ Hill climbing is efficient but can get stuck in local optima  
✅ Variants (random-restart, stochastic, beam search) reduce but don't eliminate this risk  
✅ Simulated annealing escapes local optima by occasionally accepting worse states  

### Linear Programming
✅ Optimizes a linear objective under linear constraints  
✅ Flip `≥` to `≤` by multiplying by `-1`  
✅ Use `scipy.optimize.linprog` in Python  

### Constraint Satisfaction
✅ Variables + Domains + Constraints — find an assignment satisfying all constraints  
✅ Node consistency: unary constraints satisfied for all domain values  
✅ Arc consistency (AC-3): binary constraints satisfied — prune domains propagation  
✅ Backtracking search: assign one variable at a time, undo on failure  
✅ MAC (Maintaining Arc Consistency): interleave AC-3 with backtracking for efficiency  
✅ MRV + Degree + Least Constraining Value heuristics reduce search space dramatically  

---

## 🔮 Applications

- **Logistics**: Shortest delivery routes (TSP)
- **Manufacturing**: Minimize machine time under resource constraints (LP)
- **Scheduling**: Exam timetables, job scheduling (CSP)
- **Puzzle solving**: Sudoku, crosswords (CSP + backtracking)
- **Resource allocation**: Budget optimization (LP)